# Nexus AI: Dual Domain Recommendation Engine 🍿🌸

Welcome to this comprehensive notebook! In this project, we build a **Content-Based Filtering Recommendation System** capable of recommending both **Movies** and **Anime**.

### Approach:
1. **Extraction**: Parse stringified JSON arrays (Cast, Crew, Genres) into Python lists.
2. **Feature Engineering**: Create continuous tokens for specific entities (e.g., `Studio Ghibli` -> `StudioGhibli`) to prevent the NLP model from confusing distinct concepts.
3. **Vectorization**: Utilize `CountVectorizer` to embed our combined "Tags" into a 5000-feature mathematical space.
4. **Similarity**: Use `Cosine Similarity` to calculate the exact distance between any two movies/animes.

Let's dive in!

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os

os.makedirs('models', exist_ok=True)
print("Environment initialized!")

--- 
## Part 1: Processing Movie Data (TMDB 5000)
We will load our movie databases and merge them on their common title key.

In [ ]:
movies = pd.read_csv('/kaggle/input/nexus-ai-datasets/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/nexus-ai-datasets/tmdb_5000_credits.csv')
movies = movies.merge(credits, on='title')

# Keep relevant columns for NLP metadata extraction
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)
movies.head(3)

### Helper Functions
These functions will systematically unpack the messy JSON columns into clean list formats.

In [ ]:
def convert(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            L.append(i['name'])
    except: pass
    return L

def convert3(text):
    # We only care about the top 3 actors
    L = []
    counter = 0
    try:
        for i in ast.literal_eval(text):
            if counter < 3:
                L.append(i['name'])
                counter+=1
            else: break
    except: pass
    return L

def fetch_director(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            if i['job'] == 'Director':
                L.append(i['name'])
                break
    except: pass
    return L

In [ ]:
# Apply logic to map datasets
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)

# Eliminate spaces to create unique, trackable tokens 
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

movies.head(3)

### Tag Aggregation & Vector Space Mapping
We combine all descriptive data into a single master parameter known as `tags`.

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new_movies = movies[['movie_id', 'title', 'tags']].copy()
new_movies['tags'] = new_movies['tags'].apply(lambda x: " ".join(x).lower())

print("Vectorizing Text Corpuses...")
cv = CountVectorizer(max_features=5000, stop_words='english')
movie_vectors = cv.fit_transform(new_movies['tags']).toarray()

print("Mapping Cosine Similarity Matrix...")
movie_similarity = cosine_similarity(movie_vectors)
print("Movie NLP Matrix complete. Shape:", movie_similarity.shape)

--- 
## Part 2: Processing Anime Data
We repeat a simplified variant of the process for the Anime dataset to generate its independent similarity matrix.

In [ ]:
print("--- Processing Anime Data ---")
anime = pd.read_csv('/kaggle/input/nexus-ai-datasets/anime.csv')
anime.dropna(subset=['synopsis'], inplace=True)

anime['genres_list'] = anime['genres'].apply(lambda x: str(x).replace(" ", "").split(",") if pd.notnull(x) else [])
anime['synopsis_list'] = anime['synopsis'].apply(lambda x: str(x).split() if pd.notnull(x) else [])
anime['tags'] = anime['synopsis_list'] + anime['genres_list']

new_anime = anime[['anime_id', 'title', 'tags']].copy()
new_anime['tags'] = new_anime['tags'].apply(lambda x: " ".join(x).lower())

cv_anime = CountVectorizer(max_features=5000, stop_words='english')
anime_vectors = cv_anime.fit_transform(new_anime['tags']).toarray()
anime_similarity = cosine_similarity(anime_vectors)

print("Anime NLP Matrix complete. Shape:", anime_similarity.shape)

--- 
## Part 3: Compacting & Saving the Engine
Because standard DataFrames are bloated, we use native `to_dict()` mechanisms before pickling. This drastically reduces server memory allocation when deploying the web application.

In [ ]:
# Save memory-optimized models for the Fast API web backend
pickle.dump(new_movies.to_dict(), open('models/movie_dict.pkl', 'wb'))
pickle.dump(movie_similarity, open('models/movie_similarity.pkl', 'wb'))

pickle.dump(new_anime.to_dict(), open('models/anime_dict.pkl', 'wb'))
pickle.dump(anime_similarity, open('models/anime_similarity.pkl', 'wb'))

print("Export Successful! Safe for App inference.")